# Objectives, verifiers and runtime budgets
Prerequisite: Primer-PY. Uses only the shared standard-library mechanisms and fixed authored candidate pool.

All textbook conclusions are also visible in Chapters 34–39. Original narrative/data: CC BY-SA 4.0. Code: Apache-2.0. No model training occurs.


In [1]:
from pathlib import Path
import sys
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/config/book.mjs").exists())
sys.path.insert(0, str(ROOT / "code/part-vi"))
from core import *
from run import run, verify
record = verify(run())
print("Hand-derived expectations verified; model training: none")

Hand-derived expectations verified; model training: none


In [2]:
data = fixture("sft-v1.json")
print("input target scored")
for position, (token, target, mask) in enumerate(zip(record["sft"]["inputs"], record["sft"]["targets"], record["sft"]["mask"])):
    print(position, data["vocabulary"][token], data["vocabulary"][target], mask)
print("Mean scored loss:", record["sft"]["loss"])
print("Preference audits:", record["preference"]["audits"])
print("KL:", record["preference"]["kl"], "DPO:", record["preference"]["dpo"])

input target scored
0 <bos> <system> 0
1 <system> brief 0
2 brief <eot> 0
3 <eot> <user> 0
4 <user> France? 0
5 France? <eot> 0
6 <eot> <assistant> 0
7 <assistant> Paris 1
8 Paris . 1
9 . <eot> 1
Mean scored loss: 0.9241962407465937
Preference audits: ['keep', 'reverse', 'quarantine', 'tie']
KL: 0.13081203594113697 DPO: {'margin': 1.0986122886681096, 'probability': 0.75, 'loss': 0.28768207245178096}


In [3]:
for row in record["verifier"]:
    print(row["id"], "substring", row["substring"], "exact", row["exact"], "rule", row["rule"]["accepted"], "process", row["process"])
assert verify_amount([680, 820], 750, "1,430")["accepted"]
assert not verify_amount([680, 820], 700, "1430")["accepted"]
print("A new cap changes the rule-derived answer to", reimburse([680, 820], 700))

A substring True exact True rule True process True
B substring False exact False rule False process False
C substring True exact True rule True process False
D substring False exact False rule True process True
E substring True exact False rule False process False
F substring True exact False rule False process False
A new cap changes the rule-derived answer to 1380


In [4]:
print("order method correct/4 calls fixture_tokens replay_ms")
for row in record["budget"]:
    if row["maximum"] == 4:
        print("".join(row["order"]), row["method"], row["correct"], row["calls"], row["tokens"], round(row["replay_elapsed_ms"], 6))
print("Times measure local controller replay, not language-model generation.")

order method correct/4 calls fixture_tokens replay_ms
ABC direct 2 4 40 0.031167
ABC steps 3 4 56 0.034292
ABC vote 2 12 168 0.096125
ABC search 4 10 145 0.073875
BAC direct 2 4 40 0.033625
BAC steps 2 4 56 0.086709
BAC vote 2 12 168 0.096583
BAC search 4 12 174 0.086417
CBA direct 2 4 40 0.03
CBA steps 2 4 56 0.03275
CBA vote 2 12 168 0.094959
CBA search 2 12 174 0.085
Times measure local controller replay, not language-model generation.
